In [1]:
pip install -q -U google-genai

In [2]:
from google import genai
from tqdm import tqdm
import time
import os
from google.colab import files

In [ ]:
AUTH_KEY = ""
batch_size = 100
client = genai.Client(api_key="")

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))

In [36]:
def parse(line):
    line = line.strip()
    parts = line.split('\t', 2)
    if len(parts) < 2:
      return None
    txt = parts[2] if len(parts) > 2 else ""
    return parts[0], parts[1], txt

In [ ]:
def translate_batch(sents):
    text = "\n".join(sents)

    try:
        results = client.models.generate_content(model="gemini-2.5-flash",contents=f"""
            Role: Computational Linguist & Translator.

            Task: Translate the following English sentences into Simplified Chinese for a "Coordination Inversion" dataset. 
            These sentences will later be used to test sentence embedding models by swapping the order of the clauses.

            Constraints & Guidelines:
            1. **Preserve Coordinate Structure**: The input sentences typically follow the structure "[Clause A], [Conjunction] [Clause B]". You MUST translate them into a similar Chinese structure: "[分句 A]，[连词] [分句 B]".
            2. **Explicit Conjunctions**: Do NOT omit conjunctions (like 'and', 'but', 'so', 'or'). Translate them explicitly (e.g., 'and' -> '并且/而且', 'but' -> '但是/可是', 'so' -> '所以').
            3. **Avoid Paired Conjunctions**: Try to avoid "Because... so..." (因为...所以...) or "Although... but..." (虽然...但是...) structures if the original only has one conjunction. Keep the conjunction attached to the second clause, similar to English.
            - Good: 他很累，**但是**他继续工作。(Easy to swap: 他继续工作，**但是**他很累。)
            - Bad: **虽然**他很累，**但是**他继续工作。(Harder to swap automatically without breaking grammar).
            4. **Punctuation**: Always use a Chinese comma (，) to separate the two clauses, exactly where the split happens in English.
            5. **Literalness**: Keep the sentence length and complexity as close to the original as possible.

            Input Text: {text}
            """)
        lines = results.text.strip().split('\n')
        lines = [line for line in lines if line.strip()]
        return lines
    except Exception as e:
        print(f"API Error: {e}")
        raise e

In [38]:
def main():
    with open(filename, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    data = []
    count = 0
    for line in lines:
        line2 = parse(line)
        if line2 is not None:
            data.append(line2)
        else:
            count += 1

    valid = len(data)
    print(f"Valid lines: {valid}")
    print(f"Skipped lines: {count}")
    rows = []

    for i in tqdm(range(0, valid, batch_size), desc="Translating"):
        batch = data[i : i + batch_size]
        texts = [item[2] for item in batch]

        try:
            translations = translate_batch(texts)

            for j, trans_text in enumerate(translations):
                col1 = batch[j][0]
                col2 = batch[j][1]
                rows.append(f"{col1}\t{col2}\t{trans_text}")
        except Exception:
            print("Error.")
            break

    output_filename = f"translated_{filename}"

    with open(output_filename, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(row + "\n")

    if os.path.exists(output_filename):
        print(f"File saved: {output_filename}")
        files.download(output_filename)
    else:
        print("Fail.")

In [ ]:
main()